# Knicks in Six — 01 · Data Gathering

Goal: predict the likelihood of the Knicks winning their **next game, game 6, and game 7** of the Finals with a **series-aware** model.

Project decisions:

| Decision | Choice |
|---|---|
| Live data | `nba_api` (stats.nba.com), snapshot-cached to `data/cache/*.parquet` |
| Historical data | Manually downloaded Kaggle dumps in `data/raw/` (no Kaggle API) |
| Scope | Playoffs **1980–present**, team + player level, season ids kept for era normalization |
| Model (later) | Series-aware: series score, elimination flag, rest days, home court |
| Storage | Parquet, fetch-if-missing; API failures fall back to last snapshot |

This notebook only **gathers and inventories** data. Feature engineering and modeling come in later notebooks. Reusable fetch/cache logic lives in `src/data.py`.

In [4]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src import data

pd.set_option("display.max_columns", 40)
print(f"Season: {data.CURRENT_SEASON} | Knicks team id: {data.KNICKS_ID}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Season: 2025-26 | Knicks team id: 1610612752


## 1. Live data (`nba_api`)

Current-season team strength tables. Each call writes a parquet snapshot to `data/cache/`; pass `refresh=True` to force a refetch (e.g. after a game is played).

In [5]:
team_adv = data.fetch_team_stats(season_type="Regular Season", measure="Advanced")
team_base = data.fetch_team_stats(season_type="Regular Season", measure="Base")
playoff_adv = data.fetch_team_stats(season_type="Playoffs", measure="Advanced")

cols = ["TEAM_NAME", "W", "L", "OFF_RATING", "DEF_RATING", "NET_RATING", "PACE"]
team_adv[cols].sort_values("NET_RATING", ascending=False).head(10)

,TEAM_NAME,W,L,OFF_RATING,DEF_RATING,NET_RATING,PACE
20,Oklahoma City Thunder,64,18,117.6,106.5,11.1,100.37
26,San Antonio Spurs,62,20,118.7,110.4,8.4,100.72
8,Detroit Pistons,60,22,117.3,108.9,8.4,99.88
1,Boston Celtics,56,26,120.0,111.7,8.3,95.58
19,New York Knicks,53,29,118.7,112.3,6.4,97.71
10,Houston Rockets,52,30,117.5,112.1,5.4,96.98
7,Denver Nuggets,54,28,121.2,116.0,5.2,99.49
3,Charlotte Hornets,44,38,118.4,113.5,4.9,97.59
5,Cleveland Cavaliers,52,30,118.3,114.1,4.1,100.70
17,Minnesota Timberwolves,49,33,115.6,112.5,3.1,101.50


In [6]:
playoff_log = data.fetch_league_game_log(season_type="Playoffs")
season_log = data.fetch_league_game_log(season_type="Regular Season")

print(f"Playoff team-games: {len(playoff_log)} | Regular-season team-games: {len(season_log)}")

nyk_playoffs = playoff_log[playoff_log["TEAM_ID"] == data.KNICKS_ID]
nyk_playoffs[["GAME_DATE", "MATCHUP", "WL", "PTS", "PLUS_MINUS"]].tail(8)

Playoff team-games: 168 | Regular-season team-games: 2460


,GAME_DATE,MATCHUP,WL,PTS,PLUS_MINUS
140,2026-05-19,NYK vs. CLE,W,115,11.0
144,2026-05-21,NYK vs. CLE,W,109,16.0
148,2026-05-23,NYK @ CLE,W,121,13.0
152,2026-05-25,NYK @ CLE,W,130,37.0
160,2026-06-03,NYK @ SAS,W,105,10.0
163,2026-06-05,NYK @ SAS,W,105,1.0
164,2026-06-08,NYK vs. SAS,L,111,-4.0
167,2026-06-10,NYK vs. SAS,W,107,1.0


### Current Finals series state

Derived from the playoff game log: opponent, series score, and who hosts the next game / game 6 / game 7 under the 2-2-1-1-1 format. This dict is the live input the model will score against.

In [7]:
state = data.knicks_series_state()
for k, v in state.items():
    print(f"{k:>22}: {v}")

                season: 2025-26
              opponent: SAS
         series_record: 3-1
                  wins: 3
                losses: 1
          games_played: 4
         next_game_num: 5
        knicks_has_hca: False
      knicks_home_next: False
        knicks_home_g6: True
        knicks_home_g7: False
 elimination_game_next: True
        last_game_date: 2026-06-10


## 2. Historical data (manual Kaggle dumps)

Training data for the series-aware model: playoff games and player box scores, **1980–present**. Kaggle API is intentionally not used — download the files yourself and drop them in `data/raw/` (see `data/raw/README.md` for the shopping list and candidate datasets).

The cell below inventories whatever is there and previews each file so we can map schemas in the feature-engineering step.

In [8]:
raw_files = data.list_raw_files()

if not raw_files:
    print("No raw files yet — see data/raw/README.md for what to download.")
for path in raw_files:
    head, n_rows = data.preview_raw(path)
    rel = path.relative_to(data.RAW_DIR)
    size_mb = path.stat().st_size / 1e6
    print(f"\n=== {rel} — {n_rows:,} rows x {head.shape[1]} cols ({size_mb:,.0f} MB) ===")
    print("columns:", list(head.columns)[:25])
    display(head)


=== nba_dataset/Advanced.csv — 33,339 rows x 30 cols (5 MB) ===
columns: ['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs', 'mp', 'per', 'ts_percent', 'x3p_ar', 'f_tr', 'orb_percent', 'drb_percent', 'trb_percent', 'ast_percent', 'stl_percent', 'blk_percent', 'tov_percent', 'usg_percent', 'ows', 'dws', 'ws']


,season,lg,player,player_id,age,team,pos,g,gs,mp,per,ts_percent,x3p_ar,f_tr,orb_percent,drb_percent,trb_percent,ast_percent,stl_percent,blk_percent,tov_percent,usg_percent,ows,dws,ws,ws_48,obpm,dbpm,bpm,vorp
0,2026,NBA,Precious Achiuwa,achiupr01,26,SAC,C,73,57,1745,16.2,0.558,0.162,0.232,11.0,20.8,15.8,8.5,1.8,2.6,9.0,17.6,1.8,1.2,2.9,0.081,-0.5,-1.1,-1.6,0.2
1,2026,NBA,Steven Adams,adamsst01,32,HOU,C,32,11,730,15.8,0.535,0.000,0.583,22.3,19.6,21.0,8.3,1.5,2.5,16.7,12.1,1.1,1.0,2.1,0.141,-0.1,-0.3,-0.5,0.3
2,2026,NBA,Bam Adebayo,adebaba01,28,MIA,C,73,73,2365,18.7,0.551,0.349,0.369,6.5,25.7,16.1,14.1,1.7,1.8,8.3,25.0,2.8,3.6,6.4,0.129,1.5,0.4,2.0,2.4



=== nba_dataset/All-Star Selections.csv — 2,058 rows x 6 cols (0 MB) ===
columns: ['player', 'player_id', 'team', 'season', 'lg', 'replaced']


,player,player_id,team,season,lg,replaced
0,Scottie Barnes,barnesc01,Team Stars,2026,NBA,False
1,Devin Booker,bookede01,Team Stars,2026,NBA,False
2,Cade Cunningham,cunnica01,Team Stars,2026,NBA,False



=== nba_dataset/Draft Pick History.csv — 8,383 rows x 8 cols (0 MB) ===
columns: ['season', 'lg', 'overall_pick', 'round', 'tm', 'player', 'player_id', 'college']


...

,teamId,teamCity,teamName,teamAbbrev,seasonFounded,seasonActiveTill,league
0,1610612737,Tri-Cities,Blackhawks,TRI,1946,1948,BAA
1,1610612737,Milwaukee,Hawks,MIL,1951,1954,NBA
2,1610612737,St. Louis,Hawks,STL,1955,1967,NBA



=== nba_stats/TeamStatistics.csv — 146,558 rows x 59 cols (36 MB) ===
columns: ['gameId', 'gameDateTimeEst', 'teamCity', 'teamName', 'teamId', 'opponentTeamCity', 'opponentTeamName', 'opponentTeamId', 'home', 'win', 'teamScore', 'opponentScore', 'assists', 'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade', 'fieldGoalsPercentage', 'threePointersAttempted', 'threePointersMade', 'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade', 'freeThrowsPercentage', 'reboundsDefensive']


,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,teamScore,opponentScore,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,...,pointsFastBreak,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId,gameType,gameLabel,gameSubLabel,seriesGameNumber,seed,reboundsTeam,turnoversTeam,ot1Points,ot2Points,otAllPoints,gameDate
0,42500404,2026-06-10 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,0,0,106,107,24,4,10,86,36,0.419,43,17,...,12,25,28,10,2,0,1,3,NaN,Playoffs,NBA Finals,Game 4,4,2,8,1,NaN,NaN,NaN,2026-06-10 20:30:00
1,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,1,1,107,106,23,4,6,78,36,0.462,32,15,...,13,11,34,14,2,0,3,1,NaN,Playoffs,NBA Finals,Game 4,4,3,14,2,NaN,NaN,NaN,2026-06-10 20:30:00
2,42500403,2026-06-08 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,1,0,111,115,18,6,4,88,40,0.455,37,13,...,15,7,46,21,4,0,2,1,NaN,Playoffs,NBA Finals,Game 3,3,3,10,0,NaN,NaN,NaN,2026-06-08 20:30:00



=== nba_stats/TeamStatisticsExtended.csv — 79,722 rows x 104 cols (38 MB) ===
columns: ['gameId', 'gameDateTimeEst', 'gameType', 'gameLabel', 'gameSubLabel', 'seriesGameNumber', 'teamId', 'teamCity', 'teamName', 'opponentTeamId', 'opponentTeamCity', 'opponentTeamName', 'home', 'win', 'teamScore', 'opponentScore', 'seed', 'numMinutes', 'assists', 'steals', 'blocks', 'blocksAgainst', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage']


,gameId,gameDateTimeEst,gameType,gameLabel,gameSubLabel,seriesGameNumber,teamId,teamCity,teamName,opponentTeamId,opponentTeamCity,opponentTeamName,home,win,teamScore,opponentScore,seed,numMinutes,assists,steals,...,percentFieldGoalAttempts2Point,percentFieldGoalAttempts3Point,percentPoints2Point,percentPoints2PointMidRange,percentPoints3Point,percentPointsFastBreak,percentPointsFreeThrow,percentPointsOffTurnovers,percentPointsInPaint,percentAssisted2PointMade,percentUnassisted2PointMade,percentAssisted3PointMade,percentUnassisted3PointMade,percentAssistedFieldGoalsMade,percentUnassistedFieldGoalsMade,freeThrowAttemptRate,opponentEffectiveFieldGoalPercentage,opponentFreeThrowAttemptRate,opponentTurnoverPercentage,opponentOffensiveReboundPercentage
0,42500404,2026-06-10 20:30:00,Playoffs,NBA Finals,Game 4,4,1610612759,San Antonio,Spurs,1610612752,New York,Knicks,0,0,106,107,2,240.0,24,10,...,0.500,0.500,0.358,0.094,0.481,0.113,0.160,0.236,0.264,0.421,0.579,0.941,0.059,0.667,0.333,0.233,0.558,0.359,0.161,0.283
1,42500404,2026-06-10 20:30:00,Playoffs,NBA Finals,Game 4,4,1610612752,New York,Knicks,1610612759,San Antonio,Spurs,1,1,107,106,3,240.0,23,6,...,0.590,0.410,0.393,0.075,0.421,0.121,0.187,0.103,0.318,0.381,0.619,1.000,0.000,0.639,0.361,0.359,0.517,0.233,0.128,0.308
2,42500403,2026-06-08 20:30:00,Playoffs,NBA Finals,Game 3,3,1610612759,San Antonio,Spurs,1610612752,New York,Knicks,0,1,115,111,2,240.0,28,7,...,0.595,0.405,0.470,0.087,0.313,0.043,0.217,0.183,0.383,0.630,0.370,0.917,0.083,0.718,0.282,0.381,0.528,0.250,0.140,0.373


## 3. Data inventory

Everything gathered so far, across live snapshots and raw dumps.

In [9]:
data.data_inventory()

,kind,file,size_mb
0,cache,game_log_2025_26_playoffs.parquet,0.02
1,cache,game_log_2025_26_regular_season.parquet,0.09
2,cache,team_stats_2025_26_playoffs_advanced.parquet,0.03
3,cache,team_stats_2025_26_regular_season_advanced.par...,0.03
4,cache,team_stats_2025_26_regular_season_base.parquet,0.04
5,raw,nba_dataset/Advanced.csv,4.63
6,raw,nba_dataset/All-Star Selections.csv,0.09
7,raw,nba_dataset/Draft Pick History.csv,0.45
8,raw,nba_dataset/End of Season Teams (Voting).csv,0.32
9,raw,nba_dataset/End of Season Teams.csv,0.11


## Next steps

1. Download the Kaggle dumps into `data/raw/` and re-run section 2.
2. **02_features**: derive per-game series context (game number, series score, elimination flags, rest days) from the historical games table; era-normalize team strength (1980s pace ≠ 2020s pace); aggregate player box scores into availability/star-performance features.
3. **03_model**: train the series-aware win-probability model, score game 5/6/7, and Monte-Carlo the rest of the series.